# PM2.5 Forecasting Baseline
Nachat Jatusripitak

In [4]:
# Import required packages for data management and evaluation
import pandas as pd
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [ ]:
# Load dataset as CSV
df = pd.read_csv('../dataset/dataset_1_pixels_grid_indices.csv')
df['date'] = pd.to_datetime(df['date']) 
df = df.sort_values(['row', 'col', 'date'])

## Question 1: How hard is it to predict *level* of PM2.5?

I use a persistence baseline to evaluate the difficulty of predicting the level of PM2.5 from day-to-day.<br>In particular, I predict
$\hat PM2.5_t = PM2.5_{t-1}$ and compute MSE, RMSE, MAE, and R-squared.

In [4]:
# Create the lagged column
df = df.sort_values(['row', 'col', 'date'])
df['pm25_yesterday'] = df.groupby(['row', 'col'])['pm25_today'].shift(1)
df = df.dropna()

We find significant autocorrelation between PM2.5 values from day-to-day. A naive persistence model (essentially a random walk) achieves R-squared of 0.8 with a RMSE of 8.35.

This result indicates that predicting *changes* in PM2.5 is a much more interesting (and difficult) task.

In [5]:
today = df["pm25_today"]

yesterday = df["pm25_yesterday"]

mse = mean_squared_error(today, yesterday)
rmse = root_mean_squared_error(today, yesterday)
mae = mean_absolute_error(today, yesterday)
r2 = r2_score(today, yesterday)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R^2: {r2}")

MSE: 69.67091063727962
RMSE: 8.346910244951697
MAE: 3.8036435489608267
R^2: 0.8039799205896393


# Question 2: How hard is it to predict $\Delta PM_{2.5}$?

In [21]:
# Create the lagged column
df = df.sort_values(['row', 'col', 'date'])
df['pm25_change_yesterday'] = df.groupby(['row', 'col'])['pm25_change'].shift(1)
df = df.dropna()

After differencing, we find that a naive persistence model performs poorly, as expected. We see that predicting $\Delta PM_{2.5}$ is a much harder task than predicting $PM_{2.5}$ levels alone.

In [23]:
today = df["pm25_change"]
yesterday = df["pm25_change_yesterday"]

zeros = np.zeros_like(today)

mse = mean_squared_error(today, zeros)
rmse = root_mean_squared_error(today, zeros)
mae = mean_absolute_error(today, zeros)
r2 = r2_score(today, zeros)

print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"R^2: {r2}")

MSE: 66.60426377777372
RMSE: 8.161143533707376
MAE: 3.7746982370286597
R^2: -2.72168462889244e-05


In [2]:
def rolling_window_cv(train_days, gap_days, test_days, df, model):
    dates = df["date"].unique()
    train_start = 0
    num_days = len(dates)
    i = 1
    errors = []

    while True:
        train_end = train_start + train_days
        test_start = train_end + gap_days
        test_end = test_start + test_days

        if test_end > num_days:
            break

        train_dates = dates[train_start: train_end]
        test_dates = dates[test_start: test_end]

        print(f'Fold {i}')
        print(train_dates.min(), train_dates.max())
        print(test_dates.min(), test_dates.max())

        test_data = df[df["date"].isin(test_dates)]

        y_test = test_data['delta_pm25_t+1']
        
        y_preds = np.zeros_like(y_test)

        mse = mean_squared_error(y_test, y_preds)
        rmse = root_mean_squared_error(y_test, y_preds)
        mae = mean_absolute_error(y_test, y_preds)
        r2 = r2_score(y_test, y_preds)

        errors.append(rmse)

        print(f"MSE: {mse}")
        print(f"RMSE: {rmse}")
        print(f"MAE: {mae}")
        print(f"R^2: {r2}")
        train_start += test_days
        i += 1

    print(np.mean(errors))



Fold 1
2018-07-04 00:00:00 2021-11-12 00:00:00
2021-12-03 00:00:00 2022-01-01 00:00:00


KeyError: 'delta_pm25_t+1'